# CREB5 in human AD astrocytes — Notebook 01

SEA-AD discovery and regional replication, GSE160936 independent validation, cross-region concordance, influence checks, and Hallmark enrichment.

Differential expression is performed at the donor level using raw-count pseudobulk. The notebook reuses saved checkpoints when they are present; set `REBUILD = True` to recompute them.


## Setup

In [ ]:
%pip install -q \
    "cellxgene-census==1.18.0" "pydeseq2==0.5.4" \
    "scanpy==1.11.5" "anndata==0.12.6" "pandas==2.2.3" \
    "scikit-misc==0.5.2" "harmonypy==0.0.10" \
    "igraph==1.0.0" "leidenalg>=0.10,<0.11" "gseapy==1.1.10"


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/AD_Astrocyte_Paper_01")
GSE = ROOT / "GSE160936_independent_validation"
GSE.mkdir(parents=True, exist_ok=True)

REBUILD = False


In [ ]:
import gc, gzip, io, os, shutil, tarfile
import requests
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import sparse
import anndata as ad
import scanpy as sc
import cellxgene_census

from scipy.io import mmread
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

def read_10x_gz(folder):
    folder = Path(folder)
    with gzip.open(folder / "features.tsv.gz", "rt") as f:
        features = pd.read_csv(
            f, sep="\t", header=None,
            names=["feature_id", "gene", "feature_type"]
        )
    with gzip.open(folder / "barcodes.tsv.gz", "rt") as f:
        barcodes = pd.read_csv(f, header=None)[0].astype(str).to_numpy()
    with gzip.open(folder / "matrix.mtx.gz", "rb") as f:
        X = mmread(f).tocsr()
    return X, features, barcodes

def run_deseq(counts, metadata, design, contrast):
    dds = DeseqDataSet(
        counts=counts, metadata=metadata,
        design=design, refit_cooks=True
    )
    dds.deseq2()
    stats = DeseqStats(dds, contrast=contrast)
    stats.summary()
    return stats.results_df.copy()


## SEA-AD MTG discovery

In [ ]:
meta_path = ROOT / "SEA_AD_donor_metadata.xlsx"
if REBUILD or not meta_path.exists():
    url = "https://cdn.prod.website-files.com/689cfbd308fa7373b604d290/68debdfdd1b8e9f8fd64dab0_sea-ad_cohort_donor_metadata_072524.xlsx"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    meta_path.write_bytes(r.content)

donors = pd.read_excel(meta_path)
donors["Donor ID"] = donors["Donor ID"].astype(str)

group_map = donors.set_index("Donor ID")["Overall AD neuropathological Change"].to_dict()
age_map = donors.set_index("Donor ID")["Age at Death"].to_dict()
sex_map = donors.set_index("Donor ID")["Sex"].to_dict()
rin_map = donors.set_index("Donor ID")["RIN"].to_dict()
pmi_map = donors.set_index("Donor ID")["PMI"].to_dict()
ph_map = donors.set_index("Donor ID")["Brain pH"].to_dict()

target_donors = donors.loc[
    donors["Overall AD neuropathological Change"].isin(["High", "Not AD"]),
    "Donor ID"
].tolist()

print(pd.Series([group_map[d] for d in target_donors]).value_counts())


In [ ]:
mtg_pb_path = ROOT / "SEAAD_MTG_astrocyte_pseudobulk_filtered.h5ad"
mtg_meta_path = ROOT / "SEAAD_MTG_DE_donor_metadata_QCadjusted.csv"

def build_mtg_pseudobulk():
    dataset_id = "c2876b1b-06d8-4d96-a56b-5304f815b99a"
    donor_filter = "[" + ", ".join(repr(x) for x in target_donors) + "]"
    obs_filter = (
        f"dataset_id == '{dataset_id}' and "
        "cell_type == 'astrocyte of the cerebral cortex' and "
        f"donor_id in {donor_filter}"
    )

    with cellxgene_census.open_soma(census_version="2025-11-08") as census:
        astro = cellxgene_census.get_anndata(
            census=census,
            organism="Homo sapiens",
            X_name="raw",
            obs_value_filter=obs_filter,
            obs_column_names=["dataset_id","donor_id","cell_type","sex","is_primary_data"],
            var_column_names=["feature_id","feature_name"],
        )

    donor_ids = astro.obs["donor_id"].astype(str).to_numpy()
    unique_donors, donor_index = np.unique(donor_ids, return_inverse=True)
    agg = sparse.csr_matrix(
        (np.ones(len(donor_index), dtype=np.int32),
         (donor_index, np.arange(len(donor_index)))),
        shape=(len(unique_donors), astro.n_obs),
    )

    pb_obs = pd.DataFrame(index=unique_donors)
    pb_obs["donor_id"] = unique_donors
    pb_obs["ad_group"] = pb_obs["donor_id"].map(group_map)
    pb_obs["n_nuclei"] = np.bincount(donor_index)
    pb_obs["age"] = pb_obs["donor_id"].map(age_map)
    pb_obs["sex"] = pb_obs["donor_id"].map(sex_map)
    pb_obs["RIN"] = pb_obs["donor_id"].map(rin_map)
    pb_obs["PMI"] = pb_obs["donor_id"].map(pmi_map)
    pb_obs["Brain_pH"] = pb_obs["donor_id"].map(ph_map)

    pseudobulk = ad.AnnData(X=agg @ astro.X, obs=pb_obs, var=astro.var.copy())
    lib = np.asarray(pseudobulk.X.sum(axis=1)).ravel()
    cpm = pseudobulk.X.multiply((1_000_000 / lib)[:, None])
    keep = np.asarray((cpm >= 1).sum(axis=0)).ravel() >= 9

    out = pseudobulk[:, keep].copy()
    out.write_h5ad(mtg_pb_path, compression="gzip")
    out.obs[["ad_group","age","sex","RIN","PMI","Brain_pH"]].to_csv(mtg_meta_path)
    return out


In [ ]:
if REBUILD or not (mtg_pb_path.exists() and mtg_meta_path.exists()):
    mtg_pb = build_mtg_pseudobulk()
else:
    mtg_pb = ad.read_h5ad(mtg_pb_path)

mtg_meta = pd.read_csv(mtg_meta_path, index_col=0)
print("MTG pseudobulk:", mtg_pb.shape)
print(mtg_meta["ad_group"].value_counts())


In [ ]:
mtg_out = ROOT / "SEAAD_MTG_High_vs_NotAD_DESeq2_primary_RINadjusted.csv"

if REBUILD or not mtg_out.exists():
    mtg_pb = ad.read_h5ad(mtg_pb_path)
    mtg_meta = pd.read_csv(mtg_meta_path, index_col=0)
    mtg_pb.obs_names = mtg_pb.obs_names.astype(str)
    mtg_meta.index = mtg_meta.index.astype(str)
    mtg_meta = mtg_meta.loc[mtg_pb.obs_names].copy()

    X = mtg_pb.X.toarray() if sparse.issparse(mtg_pb.X) else np.asarray(mtg_pb.X)
    gene_ids = (
        mtg_pb.var["feature_id"].astype(str).to_numpy()
        if "feature_id" in mtg_pb.var.columns
        else mtg_pb.var_names.astype(str).to_numpy()
    )
    counts = pd.DataFrame(
        np.rint(X).astype(np.int64),
        index=pd.Index(mtg_pb.obs_names, dtype=object),
        columns=pd.Index(gene_ids, dtype=object),
    )

    meta = mtg_meta[["age", "sex", "RIN", "ad_group"]].copy()
    meta["age"] = pd.to_numeric(meta["age"])
    meta["RIN"] = pd.to_numeric(meta["RIN"])
    meta["sex"] = pd.Categorical(meta["sex"].astype(str), categories=["Female", "Male"])
    meta["ad_group"] = pd.Categorical(meta["ad_group"].astype(str), categories=["Not AD", "High"])
    meta.index = counts.index

    mtg_res = run_deseq(
        counts, meta, "~ age + sex + RIN + ad_group",
        ["ad_group", "High", "Not AD"],
    )
    gene_map = dict(zip(gene_ids, mtg_pb.var["feature_name"].astype(str)))
    mtg_res["gene"] = [gene_map.get(str(x), str(x)) for x in mtg_res.index]
    mtg_res = mtg_res[["gene","baseMean","log2FoldChange","lfcSE","stat","pvalue","padj"]]
    mtg_res.to_csv(mtg_out)
    mtg_res.loc[mtg_res["padj"].notna() & (mtg_res["padj"] < 0.05)].to_csv(
        ROOT / "SEAAD_MTG_High_vs_NotAD_DESeq2_primary_RINadjusted_significant.csv"
    )
else:
    mtg_res = pd.read_csv(mtg_out, index_col=0)

print(mtg_res.loc[mtg_res["gene"].eq("CREB5")].to_string())


## SEA-AD DLPFC regional replication

In [ ]:
DLPFC_SOURCE_ID = "74d584f0-74fc-482e-b944-e76f29c1ab85"
dlpfc_source = ROOT / "SEAAD_Astrocyte_DLPFC_source.h5ad"
dlpfc_pb_path = ROOT / "SEAAD_DLPFC_astrocyte_pseudobulk_filtered.h5ad"

if REBUILD or not dlpfc_pb_path.exists():
    if not dlpfc_source.exists():
        tmp = ROOT / "SEAAD_Astrocyte_DLPFC_partial.h5ad"
        tmp.unlink(missing_ok=True)
        cellxgene_census.download_source_h5ad(
            DLPFC_SOURCE_ID,
            to_path=str(tmp),
            census_version="2025-11-08",
            progress_bar=True,
        )
        tmp.replace(dlpfc_source)

    print(dlpfc_source, round(dlpfc_source.stat().st_size / 1024**3, 2), "GB")
else:
    print("Using cached DLPFC pseudobulk:", dlpfc_pb_path.name)


In [ ]:
def aggregate_dlpfc_counts(adata, donors):
    donor_to_row = {d: i for i, d in enumerate(donors)}
    obs_donors = adata.obs["donor_id"].astype(str).to_numpy()
    counts = np.zeros((len(donors), adata.raw.shape[1]), dtype=np.float64)
    nuclei = {d: 0 for d in donors}

    for start in range(0, adata.n_obs, 5000):
        stop = min(start + 5000, adata.n_obs)
        chunk_donors = obs_donors[start:stop]
        keep = np.array([d in donor_to_row for d in chunk_donors])
        if not keep.any():
            continue

        X = adata.raw.X[start:stop, :]
        X = X if sparse.issparse(X) else sparse.csr_matrix(X)
        X = X[keep]

        kept_donors = chunk_donors[keep]
        rows = np.array([donor_to_row[d] for d in kept_donors], dtype=np.int64)
        agg = sparse.csr_matrix(
            (np.ones(len(rows)), (rows, np.arange(len(rows)))),
            shape=(len(donors), len(rows)),
        )
        counts += (agg @ X).toarray()

        d, n = np.unique(kept_donors, return_counts=True)
        for donor, count in zip(d, n):
            nuclei[donor] += int(count)

    return counts, nuclei


In [ ]:
def build_dlpfc_pseudobulk():
    local_source = Path("/content/SEAAD_Astrocyte_DLPFC_source.h5ad")
    if not local_source.exists() or local_source.stat().st_size != dlpfc_source.stat().st_size:
        shutil.copyfile(dlpfc_source, local_source)

    mtg_pb = ad.read_h5ad(mtg_pb_path)
    mtg_meta = pd.read_csv(mtg_meta_path, index_col=0)
    mtg_meta.index = mtg_meta.index.astype(str)

    mtg_gene_ids = (
        mtg_pb.var["feature_id"].astype(str).to_numpy()
        if "feature_id" in mtg_pb.var.columns
        else mtg_pb.var_names.astype(str).to_numpy()
    )

    adata = ad.read_h5ad(local_source, backed="r")
    if adata.raw is None:
        raise RuntimeError("DLPFC raw counts are missing.")

    obs_donors = adata.obs["donor_id"].astype(str).to_numpy()
    selected = set(mtg_meta.index[mtg_meta["ad_group"].isin(["High", "Not AD"])])
    available = sorted(set(obs_donors) & selected)
    if len(available) != 48:
        raise RuntimeError(f"Expected 48 DLPFC donors, found {len(available)}.")

    pb_all, nuclei_counts = aggregate_dlpfc_counts(adata, available)
    dlpfc_ids = adata.raw.var_names.astype(str).to_numpy()
    positions = pd.Index(dlpfc_ids).get_indexer(mtg_gene_ids)
    found = positions >= 0
    if found.sum() < 19000:
        raise RuntimeError("Unexpectedly low MTG/DLPFC gene overlap.")

    pb_counts = pb_all[:, positions[found]]
    pb_obs = mtg_meta.loc[available].copy()
    pb_obs["donor_id"] = pb_obs.index
    pb_obs["n_astrocyte_nuclei"] = [nuclei_counts[d] for d in available]

    pb_var = mtg_pb.var.iloc[np.where(found)[0]].copy()
    pb_var.index = mtg_gene_ids[found]
    out = ad.AnnData(X=np.rint(pb_counts).astype(np.int64), obs=pb_obs, var=pb_var)
    out.write_h5ad(dlpfc_pb_path, compression="gzip")
    adata.file.close()
    return out


In [ ]:
if REBUILD or not dlpfc_pb_path.exists():
    dlpfc_pb = build_dlpfc_pseudobulk()
else:
    dlpfc_pb = ad.read_h5ad(dlpfc_pb_path)

print("DLPFC pseudobulk:", dlpfc_pb.shape)
print(dlpfc_pb.obs["ad_group"].value_counts())


In [ ]:
dlpfc_out = ROOT / "SEAAD_DLPFC_High_vs_NotAD_DESeq2_primary_RINadjusted.csv"

if REBUILD or not dlpfc_out.exists():
    dlpfc_pb = ad.read_h5ad(dlpfc_pb_path)
    gene_ids = dlpfc_pb.var_names.astype(str).to_numpy()
    X = dlpfc_pb.X.toarray() if sparse.issparse(dlpfc_pb.X) else np.asarray(dlpfc_pb.X)

    counts = pd.DataFrame(
        np.rint(X).astype(np.int64),
        index=pd.Index(dlpfc_pb.obs_names.astype(str), dtype=object),
        columns=pd.Index(gene_ids, dtype=object),
    )

    meta = dlpfc_pb.obs[["age", "sex", "RIN", "ad_group"]].copy()
    meta["age"] = pd.to_numeric(meta["age"])
    meta["RIN"] = pd.to_numeric(meta["RIN"])
    meta["sex"] = pd.Categorical(meta["sex"].astype(str), categories=["Female", "Male"])
    meta["ad_group"] = pd.Categorical(meta["ad_group"].astype(str), categories=["Not AD", "High"])
    meta.index = counts.index

    dlpfc_res = run_deseq(
        counts, meta, "~ age + sex + RIN + ad_group",
        ["ad_group", "High", "Not AD"],
    )
    gene_map = dict(zip(gene_ids, dlpfc_pb.var["feature_name"].astype(str)))
    dlpfc_res["gene"] = [gene_map.get(str(x), str(x)) for x in dlpfc_res.index]
    dlpfc_res = dlpfc_res[["gene","baseMean","log2FoldChange","lfcSE","stat","pvalue","padj"]]
    dlpfc_res.to_csv(dlpfc_out)
    dlpfc_res.loc[dlpfc_res["padj"].notna() & (dlpfc_res["padj"] < 0.05)].to_csv(
        ROOT / "SEAAD_DLPFC_High_vs_NotAD_DESeq2_primary_RINadjusted_significant.csv"
    )
else:
    dlpfc_res = pd.read_csv(dlpfc_out, index_col=0)

print(dlpfc_res.loc[dlpfc_res["gene"].eq("CREB5")].to_string())


In [ ]:
from scipy.stats import pearsonr, spearmanr

mtg = pd.read_csv(ROOT / "SEAAD_MTG_High_vs_NotAD_DESeq2_primary_RINadjusted.csv", index_col=0)
dlpfc = pd.read_csv(ROOT / "SEAAD_DLPFC_High_vs_NotAD_DESeq2_primary_RINadjusted.csv", index_col=0)

comparison = mtg[
    ["gene", "baseMean", "log2FoldChange", "stat", "pvalue", "padj"]
].rename(columns={
    "gene": "gene_MTG",
    "baseMean": "baseMean_MTG",
    "log2FoldChange": "log2FC_MTG",
    "stat": "stat_MTG",
    "pvalue": "pvalue_MTG",
    "padj": "padj_MTG",
}).join(
    dlpfc[
        ["gene", "baseMean", "log2FoldChange", "stat", "pvalue", "padj"]
    ].rename(columns={
        "gene": "gene_DLPFC",
        "baseMean": "baseMean_DLPFC",
        "log2FoldChange": "log2FC_DLPFC",
        "stat": "stat_DLPFC",
        "pvalue": "pvalue_DLPFC",
        "padj": "padj_DLPFC",
    }),
    how="inner",
)

comparison["gene"] = comparison["gene_MTG"].fillna(comparison["gene_DLPFC"])
analysis = comparison[
    (comparison["baseMean_MTG"] >= 10)
    & (comparison["baseMean_DLPFC"] >= 10)
    & comparison["log2FC_MTG"].notna()
    & comparison["log2FC_DLPFC"].notna()
    & ~comparison["gene"].astype(str).str.startswith("MT-")
].copy()

print("Pearson:", round(pearsonr(analysis["log2FC_MTG"], analysis["log2FC_DLPFC"])[0], 3))
print("Spearman:", round(spearmanr(analysis["log2FC_MTG"], analysis["log2FC_DLPFC"])[0], 3))

ranked = analysis.assign(abs_stat_MTG=analysis["stat_MTG"].abs()).sort_values("abs_stat_MTG", ascending=False)
for n in (100, 500, 1000):
    x = ranked.head(n)
    agree = (np.sign(x["log2FC_MTG"]) == np.sign(x["log2FC_DLPFC"])).mean()
    print(f"Top {n} direction agreement: {agree:.1%}")

comparison.to_csv(ROOT / "SEAAD_MTG_vs_DLPFC_gene_concordance.csv")


## GSE160936 independent validation

In [ ]:
raw_tar = GSE / "GSE160936_RAW.tar"

if not raw_tar.exists():
    url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE160nnn/GSE160936/suppl/GSE160936_RAW.tar"
    !wget -c --show-progress -O "$raw_tar" "$url"

print(f"{raw_tar.name}: {raw_tar.stat().st_size / 1024**3:.2f} GB")


In [ ]:
manifest_path = GSE / "GSE160936_verified_sample_manifest.csv"

if REBUILD or not manifest_path.exists():
    url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE160nnn/GSE160936/soft/GSE160936_family.soft.gz"
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    text = gzip.decompress(r.content).decode("utf-8", errors="replace")

    samples, current = [], None
    for line in text.splitlines():
        if line.startswith("^SAMPLE = "):
            if current is not None:
                samples.append(current)
            current = {"GSM": line.split("=", 1)[1].strip()}
        elif current is not None and line.startswith("!Sample_title = "):
            current["title"] = line.split("=", 1)[1].strip()
        elif current is not None and line.startswith("!Sample_description = "):
            current["description"] = line.split("=", 1)[1].strip()
        elif current is not None and line.startswith("!Sample_characteristics_ch1 = "):
            value = line.split("=", 1)[1].strip()
            if ":" in value:
                key, value = value.split(":", 1)
                current[key.strip().lower()] = value.strip()
    if current is not None:
        samples.append(current)

    raw_meta = pd.DataFrame(samples)
    manifest = pd.DataFrame({
        "GSM": raw_meta["GSM"],
        "title": raw_meta["title"],
        "donor": raw_meta["title"].astype(str).str.rsplit(" ", n=1).str[0],
        "disease": raw_meta["disease state"],
        "region": raw_meta["brain region (somatosensory or entorhinal cortex)"],
        "age": pd.to_numeric(raw_meta["age"], errors="coerce"),
        "sex": raw_meta["sex"],
        "RIN": pd.to_numeric(raw_meta["rin"], errors="coerce"),
        "Braak": pd.to_numeric(raw_meta["braak tangle stage"], errors="coerce"),
        "pTau": pd.to_numeric(raw_meta["ptau immunostaining (%ptau positive cells)"], errors="coerce"),
        "amyloid": pd.to_numeric(raw_meta["amyloid immunostaining (%area stained)"], errors="coerce"),
        "description": raw_meta["description"],
    })
    manifest.to_csv(manifest_path, index=False)
else:
    manifest = pd.read_csv(manifest_path)

print("Samples:", len(manifest), "| donors:", manifest["donor"].nunique())
print(pd.crosstab(manifest["disease"], manifest["region"]))


In [ ]:
matrix_root = GSE / "GSE160936_matrices"

def matrices_complete(root):
    folders = sorted(root.glob("GSM*")) if root.exists() else []
    required = ("matrix.mtx.gz", "features.tsv.gz", "barcodes.tsv.gz")
    return len(folders) == 24 and all(
        all((folder / f).exists() for f in required) for folder in folders
    )

if not matrices_complete(matrix_root):
    if matrix_root.exists():
        shutil.rmtree(matrix_root)
    matrix_root.mkdir(parents=True)

    with tarfile.open(raw_tar, "r") as outer:
        nested = [m for m in outer.getmembers() if m.isfile() and m.name.endswith(".tar.gz")]
        if len(nested) != 24:
            raise RuntimeError(f"Expected 24 sample archives, found {len(nested)}.")

        for member in nested:
            gsm = Path(member.name).name.split("_")[0]
            folder = matrix_root / gsm
            folder.mkdir()

            data = outer.extractfile(member).read()
            with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as inner:
                members = {Path(x.name).name: x for x in inner.getmembers() if x.isfile()}
                for name in ("matrix.mtx.gz", "features.tsv.gz", "barcodes.tsv.gz"):
                    src = inner.extractfile(members[name])
                    with open(folder / name, "wb") as dst:
                        shutil.copyfileobj(src, dst)

print("Complete sample folders:", len(list(matrix_root.glob("GSM*"))))


### Cell QC and clustering gene universe

In [ ]:
qc_path = GSE / "GSE160936_QC_retained_barcodes.csv"
qc_summary_path = GSE / "GSE160936_QC_summary.csv"

if REBUILD or not (qc_path.exists() and qc_summary_path.exists()):
    first = sorted(matrix_root.glob("GSM*"))[0]
    _, features, _ = read_10x_gz(first)
    mt = features["gene"].astype(str).str.startswith("MT-").to_numpy()

    rows, kept = [], []
    for folder in sorted(matrix_root.glob("GSM*")):
        gsm = folder.name
        X, _, barcodes = read_10x_gz(folder)

        total = np.asarray(X.sum(axis=0)).ravel()
        n_genes = np.asarray(X.getnnz(axis=0)).ravel()
        mt_counts = np.asarray(X[mt, :].sum(axis=0)).ravel()
        pct_mt = np.divide(mt_counts * 100, total, out=np.zeros_like(mt_counts, dtype=float), where=total > 0)

        ok = (n_genes >= 200) & (n_genes <= 6000) & (total <= 25000) & (pct_mt <= 5)

        rows.append({
            "GSM": gsm, "before_QC": len(barcodes), "after_QC": int(ok.sum()),
            "removed": int((~ok).sum()), "retained_pct": float(ok.mean() * 100),
        })
        kept.append(pd.DataFrame({
            "GSM": gsm, "barcode": barcodes[ok],
            "n_genes": n_genes[ok], "total_counts": total[ok], "pct_mito": pct_mt[ok],
        }))

    qc_summary = pd.DataFrame(rows).merge(
        manifest[["GSM", "donor", "disease", "region"]], on="GSM", how="left"
    )
    retained = pd.concat(kept, ignore_index=True).merge(
        manifest[["GSM", "donor", "disease", "region", "age", "sex", "RIN"]],
        on="GSM", how="left"
    )
    qc_summary.to_csv(qc_summary_path, index=False)
    retained.to_csv(qc_path, index=False)
else:
    qc_summary = pd.read_csv(qc_summary_path)
    retained = pd.read_csv(qc_path)

print("QC-passed nuclei:", len(retained))
print(pd.pivot_table(qc_summary, index="disease", columns="region", values="after_QC", aggfunc="sum"))


In [ ]:
gene_filter_path = GSE / "GSE160936_gene_filter_published_QC.csv"

if REBUILD or not gene_filter_path.exists():
    first = sorted(matrix_root.glob("GSM*"))[0]
    _, features, _ = read_10x_gz(first)
    detection = np.zeros(len(features), dtype=np.int64)

    for folder in sorted(matrix_root.glob("GSM*")):
        gsm = folder.name
        X, _, barcodes = read_10x_gz(folder)
        qc_set = set(retained.loc[retained["GSM"].eq(gsm), "barcode"].astype(str))
        keep_cells = np.array([b in qc_set for b in barcodes])
        detection += np.asarray(X[:, keep_cells].getnnz(axis=1)).ravel()

    mt = features["gene"].astype(str).str.startswith("MT-").to_numpy()
    gene_filter = features.copy()
    gene_filter["n_nuclei_expressed"] = detection
    gene_filter["mitochondrial"] = mt
    gene_filter["keep_for_clustering"] = (detection >= 3) & ~mt
    gene_filter.to_csv(gene_filter_path, index=False)
else:
    gene_filter = pd.read_csv(gene_filter_path)

keep_global = gene_filter["keep_for_clustering"].astype(str).str.lower().eq("true").to_numpy()
print("Genes retained for clustering:", int(keep_global.sum()))


### Consensus HVGs

In [ ]:
hvg_per_sample_path = GSE / "GSE160936_VST_sampleFiltered_HVGs_per_sample.csv"
hvg_summary_path = GSE / "GSE160936_VST_sampleFiltered_summary.csv"

if REBUILD or not (hvg_per_sample_path.exists() and hvg_summary_path.exists()):
    feature_ids = gene_filter.loc[keep_global, "feature_id"].astype(str).to_numpy()
    gene_names = gene_filter.loc[keep_global, "gene"].astype(str).to_numpy()

    hvg_rows, summary_rows = [], []
    for gsm in sorted(manifest["GSM"].astype(str)):
        folder = matrix_root / gsm
        X, _, barcodes = read_10x_gz(folder)

        qc_set = set(retained.loc[retained["GSM"].eq(gsm), "barcode"].astype(str))
        keep_cells = np.array([b in qc_set for b in barcodes])
        X = X[keep_global, :][:, keep_cells]

        detected = np.asarray(X.getnnz(axis=1)).ravel()
        gene_ok = detected >= 3
        a = ad.AnnData(X=X[gene_ok, :].T.tocsr())
        a.var_names = feature_ids[gene_ok]
        a.var["gene"] = gene_names[gene_ok]

        sc.pp.highly_variable_genes(
            a, flavor="seurat_v3", n_top_genes=2000,
            span=0.3, subset=False, inplace=True, check_values=True
        )

        selected = a.var.loc[a.var["highly_variable"]].copy()
        selected["feature_id"] = selected.index.astype(str)
        selected["vst_rank"] = selected["highly_variable_rank"].astype(float) + 1
        selected["GSM"] = gsm

        meta = manifest.loc[manifest["GSM"].eq(gsm)].iloc[0]
        for col in ("donor", "disease", "region"):
            selected[col] = meta[col]

        hvg_rows.append(selected[
            ["GSM", "donor", "disease", "region", "feature_id", "gene",
             "vst_rank", "means", "variances", "variances_norm"]
        ])
        summary_rows.append({
            "GSM": gsm, "QC_nuclei": a.n_obs,
            "global_genes": int(keep_global.sum()),
            "genes_entering_VST": a.n_vars, "HVGs": int(selected.shape[0]),
        })

    per_sample_hvg = pd.concat(hvg_rows, ignore_index=True)
    hvg_summary = pd.DataFrame(summary_rows)
    per_sample_hvg.to_csv(hvg_per_sample_path, index=False)
    hvg_summary.to_csv(hvg_summary_path, index=False)
else:
    per_sample_hvg = pd.read_csv(hvg_per_sample_path)
    hvg_summary = pd.read_csv(hvg_summary_path)

print("Per-sample HVG rows:", len(per_sample_hvg))


In [ ]:
consensus_path = GSE / "GSE160936_VST_sampleFiltered_consensus_2000.csv"

if REBUILD or not consensus_path.exists():
    consensus = (
        per_sample_hvg.groupby("feature_id", as_index=False)
        .agg(
            gene=("gene", "first"),
            n_samples_selected=("GSM", "nunique"),
            median_vst_rank=("vst_rank", "median"),
        )
        .sort_values(
            ["n_samples_selected", "median_vst_rank", "feature_id"],
            ascending=[False, True, True], kind="mergesort"
        )
        .reset_index(drop=True)
    )

    boundary = consensus.iloc[1999]
    tied = consensus[
        consensus["n_samples_selected"].eq(boundary["n_samples_selected"])
        & np.isclose(consensus["median_vst_rank"], boundary["median_vst_rank"], atol=1e-12)
    ]
    if len(tied) != 1:
        raise RuntimeError("The 2,000-feature consensus boundary is tied.")

    final_hvg = consensus.iloc[:2000].copy()
    final_hvg["integration_feature_rank"] = np.arange(1, 2001)
    final_hvg.to_csv(consensus_path, index=False)
else:
    final_hvg = pd.read_csv(consensus_path)

print("Consensus HVGs:", len(final_hvg))


### PCA and Harmony

In [ ]:
raw2000_path = GSE / "GSE160936_CORRECTED_raw2000_with_fullLibrary.h5ad"

if REBUILD or not raw2000_path.exists():
    global_ids = gene_filter.loc[keep_global, "feature_id"].astype(str).to_numpy()
    global_pos = {g: i for i, g in enumerate(global_ids)}
    hvg_ids = final_hvg["feature_id"].astype(str).to_numpy()
    hvg_pos = np.array([global_pos[g] for g in hvg_ids], dtype=int)

    matrices, obs_parts = [], []
    for gsm in sorted(manifest["GSM"].astype(str)):
        X, _, barcodes = read_10x_gz(matrix_root / gsm)
        sample_qc = retained.loc[retained["GSM"].eq(gsm)].copy()
        qc_set = set(sample_qc["barcode"].astype(str))
        keep_cells = np.array([b in qc_set for b in barcodes])

        X_full = X[keep_global, :][:, keep_cells]
        library = np.asarray(X_full.sum(axis=0)).ravel()
        n_genes = np.asarray(X_full.getnnz(axis=0)).ravel()
        matrices.append(X_full[hvg_pos, :].T.tocsr().astype(np.int32))

        meta = manifest.loc[manifest["GSM"].eq(gsm)].iloc[0]
        kept_barcodes = barcodes[keep_cells]
        obs_parts.append(pd.DataFrame({
            "cell_id": [f"{gsm}_{b}" for b in kept_barcodes],
            "GSM": gsm,
            "barcode": kept_barcodes,
            "donor": meta["donor"],
            "disease": meta["disease"],
            "region": meta["region"],
            "age": meta["age"],
            "sex": meta["sex"],
            "RIN": meta["RIN"],
            "library_size_37734_nonMT": library,
            "n_genes_detected_37734": n_genes,
        }))

    X = sp.vstack(matrices, format="csr")
    obs = pd.concat(obs_parts, ignore_index=True).set_index("cell_id", drop=False)
    var = final_hvg.copy()
    var.index = var["feature_id"].astype(str)

    if X.shape != (91620, 2000):
        raise RuntimeError(f"Unexpected clustering matrix shape: {X.shape}")

    raw2000 = ad.AnnData(X=X, obs=obs, var=var)
    raw2000.uns["normalization_denominator"] = "37,734 retained non-mitochondrial genes"
    raw2000.write_h5ad(raw2000_path, compression="gzip")
else:
    raw2000 = ad.read_h5ad(raw2000_path)

print("Clustering checkpoint:", raw2000.shape)


In [ ]:
pca_path = GSE / "GSE160936_CORRECTED_PCA_checkpoint.h5ad"
loadings_path = GSE / "GSE160936_CORRECTED_PCA_loadings.csv"

if REBUILD or not pca_path.exists():
    a = ad.read_h5ad(raw2000_path)
    library = a.obs["library_size_37734_nonMT"].to_numpy(dtype=float)

    X = a.X.tocsr().astype(np.float32)
    Xn = (sp.diags((10000 / library).astype(np.float32)) @ X).tocsr()
    Xn.data = np.log1p(Xn.data).astype(np.float32)

    Xs = Xn.toarray().astype(np.float32)
    mean = Xs.mean(axis=0, dtype=np.float64)
    sd = Xs.std(axis=0, ddof=1, dtype=np.float64)
    sd[(~np.isfinite(sd)) | (sd == 0)] = 1
    Xs -= mean.astype(np.float32)
    Xs /= sd.astype(np.float32)
    np.clip(Xs, -10, 10, out=Xs)

    pca = PCA(n_components=30, svd_solver="randomized", random_state=42)
    X_pca = pca.fit_transform(Xs).astype(np.float32)

    loadings = pd.DataFrame({
        "feature_id": a.var["feature_id"].astype(str).to_numpy(),
        "gene": a.var["gene"].astype(str).to_numpy(),
    })
    for i in range(30):
        loadings[f"PC{i+1}_loading"] = pca.components_[i]
    loadings.to_csv(loadings_path, index=False)

    pca_ckpt = ad.AnnData(
        X=sp.csr_matrix((a.n_obs, 1), dtype=np.float32),
        obs=a.obs.copy(),
        var=pd.DataFrame(index=["placeholder"]),
    )
    pca_ckpt.obsm["X_pca"] = X_pca
    pca_ckpt.uns["pca_explained_variance_ratio"] = pca.explained_variance_ratio_.astype(np.float32)
    pca_ckpt.write_h5ad(pca_path, compression="gzip")
else:
    pca_ckpt = ad.read_h5ad(pca_path)

print("PCA:", pca_ckpt.obsm["X_pca"].shape)


In [ ]:
import harmonypy as hm

pca_path = GSE / "GSE160936_CORRECTED_PCA_checkpoint.h5ad"
harmony_path = GSE / "GSE160936_CORRECTED_Harmony15_checkpoint.h5ad"

if REBUILD or not harmony_path.exists():
    pca = ad.read_h5ad(pca_path)
    X15 = np.asarray(pca.obsm["X_pca"][:, :15], dtype=np.float64)
    obs = pca.obs.copy()

    ho = hm.run_harmony(
        data_mat=X15,
        meta_data=obs,
        vars_use="GSM",
        theta=2.0,
        lamb=1.0,
        sigma=0.1,
        nclust=None,
        tau=0,
        block_size=0.05,
        max_iter_harmony=10,
        max_iter_kmeans=20,
        epsilon_cluster=1e-5,
        epsilon_harmony=1e-4,
        plot_convergence=False,
        verbose=True,
        random_state=42,
    )

    X_harmony = np.asarray(ho.Z_corr).T
    if X_harmony.shape != X15.shape:
        raise RuntimeError(f"Unexpected Harmony shape: {X_harmony.shape}")

    h = ad.AnnData(obs=obs.copy())
    h.obsm["X_pca15"] = X15.astype(np.float32)
    h.obsm["X_harmony15"] = X_harmony.astype(np.float32)
    h.write_h5ad(harmony_path, compression="gzip")
else:
    h = ad.read_h5ad(harmony_path)

print("Harmony checkpoint:", h.shape, h.obsm["X_harmony15"].shape)


### Leiden clustering and astrocyte annotation

In [ ]:
cluster_path = GSE / "GSE160936_Harmony15_Leiden05_clusters.h5ad"

if REBUILD or not cluster_path.exists():
    h = ad.read_h5ad(GSE / "GSE160936_CORRECTED_Harmony15_checkpoint.h5ad")
    sc.pp.neighbors(h, use_rep="X_harmony15", n_neighbors=20, metric="euclidean", random_state=42)
    sc.tl.leiden(
        h, resolution=0.5, random_state=42, key_added="leiden_05",
        flavor="igraph", n_iterations=2, directed=False, use_weights=True
    )
    h.write_h5ad(cluster_path, compression="gzip")
else:
    h = ad.read_h5ad(cluster_path)

print(h.obs["leiden_05"].value_counts().sort_index(key=lambda x: x.astype(int)))


In [ ]:
marker_sets = {
    "Astrocyte": ["AQP4","ALDH1L1","SLC1A2","SLC1A3","GLUL","GJA1","SOX9","FGFR3","GFAP"],
    "Microglia": ["P2RY12","TMEM119","C1QA","C1QB","C1QC","AIF1","TYROBP","CSF1R"],
    "Oligodendrocyte": ["PLP1","MBP","MOG","MOBP","MAG","OPALIN"],
    "OPC": ["PDGFRA","CSPG4","VCAN","BCAN"],
    "Neuron_general": ["SNAP25","SYT1","RBFOX3"],
    "Excitatory_neuron": ["SLC17A7","CAMK2A","SATB2"],
    "Inhibitory_neuron": ["GAD1","GAD2","SLC6A1"],
    "Endothelial": ["CLDN5","VWF","PECAM1","FLT1","KDR","EMCN"],
    "Pericyte_mural": ["RGS5","PDGFRB","NOTCH3","MCAM","CSPG4"],
    "VLMC_fibroblast": ["COL1A1","COL1A2","COL3A1","DCN","LUM"],
    "T_NK": ["CD3D","CD3E","TRBC1","TRBC2","NKG7","KLRD1"],
}


In [ ]:
def cluster_marker_tables(raw_path, cluster_path):
    raw = ad.read_h5ad(raw_path)
    clusters = ad.read_h5ad(cluster_path)
    if not raw.obs_names.equals(clusters.obs_names):
        raise RuntimeError("Raw and clustering checkpoints have different cell order.")

    raw.obs["leiden_05"] = clusters.obs["leiden_05"].astype(str).to_numpy()
    library = raw.obs["library_size_37734_nonMT"].to_numpy(dtype=float)
    Xn = (sp.diags((10000 / library).astype(np.float32)) @ raw.X.tocsr()).tocsr()
    Xn.data = np.log1p(Xn.data)

    genes = raw.var["gene"].astype(str).to_numpy()
    cluster_ids = sorted(raw.obs["leiden_05"].unique(), key=int)
    markers = sorted(set(sum(marker_sets.values(), [])))
    mean_rows, pct_rows = [], []

    for cluster in cluster_ids:
        mask = raw.obs["leiden_05"].eq(cluster).to_numpy()
        mean_row, pct_row = {"cluster": cluster}, {"cluster": cluster}

        for marker in markers:
            cols = np.where(genes == marker)[0]
            if len(cols) == 0:
                mean_row[marker] = np.nan
                pct_row[marker] = np.nan
                continue
            mean_row[marker] = float(np.asarray(Xn[mask][:, cols].sum(axis=1)).mean())
            pct_row[marker] = float(
                (np.asarray(raw.X[mask][:, cols].sum(axis=1)).ravel() > 0).mean() * 100
            )

        mean_rows.append(mean_row)
        pct_rows.append(pct_row)

    means = pd.DataFrame(mean_rows).set_index("cluster")
    pct = pd.DataFrame(pct_rows).set_index("cluster")
    scores = pd.DataFrame(index=means.index)

    for lineage, markers in marker_sets.items():
        available = [m for m in markers if m in means and means[m].notna().any()]
        z = means[available].apply(lambda x: (x - x.mean()) / x.std(ddof=0), axis=0)
        scores[lineage] = z.mean(axis=1)

    return means, pct, scores


In [ ]:
marker_mean_path = GSE / "GSE160936_Leiden05_marker_mean_expression.csv"
marker_pct_path = GSE / "GSE160936_Leiden05_marker_pct_detected.csv"
lineage_path = GSE / "GSE160936_Leiden05_lineage_scores.csv"

if REBUILD or not (marker_mean_path.exists() and marker_pct_path.exists() and lineage_path.exists()):
    marker_mean, marker_pct, scores = cluster_marker_tables(raw2000_path, cluster_path)
    marker_mean.to_csv(marker_mean_path)
    marker_pct.to_csv(marker_pct_path)
    scores.to_csv(lineage_path)
else:
    marker_mean = pd.read_csv(marker_mean_path, index_col=0)
    marker_pct = pd.read_csv(marker_pct_path, index_col=0)
    scores = pd.read_csv(lineage_path, index_col=0)

print(scores.round(3).to_string())


In [ ]:
barcode_path = GSE / "GSE160936_FINAL_astrocyte_barcodes_clusters0to4.csv"

if REBUILD or not barcode_path.exists():
    raw = ad.read_h5ad(raw2000_path)
    clusters = ad.read_h5ad(cluster_path)
    raw.obs["leiden_05"] = clusters.obs["leiden_05"].astype(str).to_numpy()

    astro_clusters = {"0", "1", "2", "3", "4"}
    astro_obs = raw.obs.loc[raw.obs["leiden_05"].astype(str).isin(astro_clusters)].copy()

    barcode_manifest = astro_obs[
        ["GSM","barcode","donor","disease","region","age","sex","RIN","leiden_05"]
    ].reset_index(names="cell_id")
    barcode_manifest.to_csv(barcode_path, index=False)

    sample_counts = (
        barcode_manifest.groupby(["GSM","donor","disease","region"], as_index=False)
        .size().rename(columns={"size":"n_astrocytes"})
    )
    sample_counts.to_csv(GSE / "GSE160936_FINAL_astrocyte_counts_by_sample.csv", index=False)
else:
    barcode_manifest = pd.read_csv(barcode_path)
    sample_counts = (
        barcode_manifest.groupby(["GSM","donor","disease","region"], as_index=False)
        .size().rename(columns={"size":"n_astrocytes"})
    )

print("Accepted astrocytes:", len(barcode_manifest))
print(sample_counts.to_string(index=False))


### Astrocyte pseudobulk

In [ ]:
pb_path = GSE / "GSE160936_FINAL_astrocyte_pseudobulk_24samples_raw.h5ad"

if REBUILD or not pb_path.exists():
    astro = pd.read_csv(GSE / "GSE160936_FINAL_astrocyte_barcodes_clusters0to4.csv")
    matrices, obs_rows = [], []

    first_X, first_features, _ = read_10x_gz(matrix_root / sorted(manifest["GSM"].astype(str))[0])
    var = first_features.copy()
    var["feature_id_unversioned"] = var["feature_id"].astype(str).str.split(".").str[0]
    var.index = var["feature_id"].astype(str)

    for gsm in sorted(manifest["GSM"].astype(str)):
        X, features, barcodes = read_10x_gz(matrix_root / gsm)
        if not features["feature_id"].astype(str).equals(first_features["feature_id"].astype(str)):
            raise RuntimeError(f"{gsm}: feature order differs from other samples.")

        wanted = set(astro.loc[astro["GSM"].eq(gsm), "barcode"].astype(str))
        keep = np.array([b in wanted for b in barcodes])
        if int(keep.sum()) != len(wanted):
            raise RuntimeError(f"{gsm}: astrocyte barcode mismatch.")

        counts = np.asarray(X[:, keep].sum(axis=1)).ravel()
        matrices.append(sp.csr_matrix(counts.reshape(1, -1)))

        meta = manifest.loc[manifest["GSM"].eq(gsm)].iloc[0]
        obs_rows.append({
            "GSM": gsm, "donor": meta["donor"], "disease": meta["disease"],
            "region": meta["region"], "age": meta["age"], "sex": meta["sex"],
            "RIN": meta["RIN"], "n_astrocytes": int(keep.sum()),
        })

    pb = ad.AnnData(
        X=sp.vstack(matrices, format="csr"),
        obs=pd.DataFrame(obs_rows).set_index("GSM"),
        var=var,
    )
    pb.write_h5ad(pb_path, compression="gzip")
else:
    pb = ad.read_h5ad(pb_path)

print("Raw astrocyte pseudobulk:", pb.shape)
print(pb.obs.groupby(["disease","region"])["n_astrocytes"].sum())


In [ ]:
de_universe_path = GSE / "GSE160936_FINAL_common_nonMT_DE_universe.h5ad"

if REBUILD or not de_universe_path.exists():
    pb = ad.read_h5ad(pb_path)
    X = pb.X.tocsr()
    ec = pb.obs["region"].eq("EC").to_numpy()
    ssc = pb.obs["region"].eq("SSC").to_numpy()

    ec_keep = np.asarray((X[ec] >= 10).sum(axis=0)).ravel() >= 6
    ssc_keep = np.asarray((X[ssc] >= 10).sum(axis=0)).ravel() >= 6
    non_mt = ~pb.var["gene"].astype(str).str.startswith("MT-").to_numpy()
    keep = ec_keep & ssc_keep & non_mt

    de_pb = pb[:, keep].copy()
    de_pb.write_h5ad(de_universe_path, compression="gzip")
    de_pb.var.to_csv(GSE / "GSE160936_FINAL_common_nonMT_DE_gene_universe.csv")
else:
    de_pb = ad.read_h5ad(de_universe_path)

print("Frozen DE universe:", de_pb.shape)


### Regional differential expression

In [ ]:
def regional_de(region):
    out = GSE / f"GSE160936_{region}_AD_vs_Control_DESeq2_primary_RINadjusted.csv"
    if not REBUILD and out.exists():
        return pd.read_csv(out, index_col=0)

    a = ad.read_h5ad(de_universe_path)
    a = a[a.obs["region"].eq(region)].copy()
    X = a.X.toarray() if sp.issparse(a.X) else np.asarray(a.X)

    gene_ids = a.var["feature_id_unversioned"].astype(str).to_numpy()
    if len(np.unique(gene_ids)) != len(gene_ids):
        raise RuntimeError("Unversioned Ensembl IDs are not unique.")

    counts = pd.DataFrame(
        np.rint(X).astype(np.int64),
        index=pd.Index(a.obs_names.astype(str), dtype=object),
        columns=pd.Index(gene_ids, dtype=object),
    )
    meta = a.obs[["RIN", "disease"]].copy()
    meta["RIN"] = pd.to_numeric(meta["RIN"])
    meta["disease"] = pd.Categorical(
        meta["disease"].astype(str),
        categories=["Non-disease control", "AD"],
    )
    meta.index = counts.index

    res = run_deseq(
        counts, meta, "~ RIN + disease",
        ["disease", "AD", "Non-disease control"],
    )
    res["gene"] = a.var["gene"].astype(str).to_numpy()
    res["feature_id_versioned"] = a.var["feature_id"].astype(str).to_numpy()
    res = res[
        ["gene","feature_id_versioned","baseMean","log2FoldChange",
         "lfcSE","stat","pvalue","padj"]
    ]
    res.to_csv(out)
    res.loc[res["padj"].notna() & (res["padj"] < 0.05)].to_csv(
        GSE / f"GSE160936_{region}_AD_vs_Control_DESeq2_primary_RINadjusted_significant.csv"
    )
    return res

ec_res = regional_de("EC")
ssc_res = regional_de("SSC")

for region, res in [("EC", ec_res), ("SSC", ssc_res)]:
    print("\n", region)
    print(res.loc[res["gene"].eq("CREB5")].to_string())


## Cross-region and cross-cohort concordance

In [ ]:
mtg = pd.read_csv(ROOT / "SEAAD_MTG_High_vs_NotAD_DESeq2_primary_RINadjusted.csv", index_col=0)
dlpfc = pd.read_csv(ROOT / "SEAAD_DLPFC_High_vs_NotAD_DESeq2_primary_RINadjusted.csv", index_col=0)
ec = pd.read_csv(GSE / "GSE160936_EC_AD_vs_Control_DESeq2_primary_RINadjusted.csv", index_col=0)
ssc = pd.read_csv(GSE / "GSE160936_SSC_AD_vs_Control_DESeq2_primary_RINadjusted.csv", index_col=0)

def prefix(df, label):
    x = df.copy()
    x.index = x.index.astype(str).str.split(".").str[0]
    return x.add_prefix(f"{label}_")

four = prefix(mtg, "MTG").join(prefix(dlpfc, "DLPFC"), how="inner")
four = four.join(prefix(ec, "EC"), how="inner").join(prefix(ssc, "SSC"), how="inner")
four = four.replace([np.inf, -np.inf], np.nan)
four = four.dropna(subset=[
    "MTG_log2FoldChange","DLPFC_log2FoldChange",
    "EC_log2FoldChange","SSC_log2FoldChange"
])

print("Common genes:", len(four))

pairs = []
for source in ("MTG", "DLPFC"):
    for target in ("EC", "SSC"):
        x, y = four[f"{source}_log2FoldChange"], four[f"{target}_log2FoldChange"]
        pairs.append({
            "source": source, "target": target, "n_genes": len(four),
            "pearson_r": pearsonr(x, y)[0],
            "spearman_rho": spearmanr(x, y)[0],
            "direction_pct": (np.sign(x) == np.sign(y)).mean() * 100,
        })
pairwise = pd.DataFrame(pairs)
print(pairwise.round(4).to_string(index=False))

four["SEAAD_mean_LFC"] = four[["MTG_log2FoldChange","DLPFC_log2FoldChange"]].mean(axis=1)
four["GSE_mean_LFC"] = four[["EC_log2FoldChange","SSC_log2FoldChange"]].mean(axis=1)

print("Cohort-mean Pearson:", round(pearsonr(four["SEAAD_mean_LFC"], four["GSE_mean_LFC"])[0], 4))
print("Cohort-mean Spearman:", round(spearmanr(four["SEAAD_mean_LFC"], four["GSE_mean_LFC"])[0], 4))

same4 = (
    np.sign(four["MTG_log2FoldChange"]).eq(np.sign(four["DLPFC_log2FoldChange"]))
    & np.sign(four["MTG_log2FoldChange"]).eq(np.sign(four["EC_log2FoldChange"]))
    & np.sign(four["MTG_log2FoldChange"]).eq(np.sign(four["SSC_log2FoldChange"]))
)
print("Same direction in all four:", int(same4.sum()), f"({same4.mean():.2%})")

four.to_csv(GSE / "SEAAD_vs_GSE160936_four_region_primary_concordance.csv")
pairwise.to_csv(GSE / "SEAAD_vs_GSE160936_pairwise_concordance_summary.csv", index=False)


In [ ]:
# CREB5 and strongest SEA-AD consensus effects
creb5_id = "ENSG00000146592"
cols = [
    "MTG_log2FoldChange","MTG_pvalue","MTG_padj",
    "DLPFC_log2FoldChange","DLPFC_pvalue","DLPFC_padj",
    "EC_log2FoldChange","EC_pvalue","EC_padj",
    "SSC_log2FoldChange","SSC_pvalue","SSC_padj",
]
print(four.loc[[creb5_id], cols].T)

ranked = four.assign(
    SEAAD_strength=(four["MTG_stat"].abs() + four["DLPFC_stat"].abs()) / 2
).sort_values("SEAAD_strength", ascending=False)

for n in (100, 500, 1000):
    x = ranked.head(n)
    both = (
        np.sign(x["SEAAD_mean_LFC"]).eq(np.sign(x["EC_log2FoldChange"]))
        & np.sign(x["SEAAD_mean_LFC"]).eq(np.sign(x["SSC_log2FoldChange"]))
    ).mean()
    print(f"Top {n} SEA-AD consensus, same direction in both GSE regions: {both:.1%}")


## Hallmark enrichment

In [ ]:
import gseapy as gp

symbol_cols = ["MTG_gene","DLPFC_gene","EC_gene","SSC_gene"]
common = four.dropna(subset=symbol_cols).copy()

same_symbol = (
    common["MTG_gene"].astype(str).eq(common["DLPFC_gene"].astype(str))
    & common["MTG_gene"].astype(str).eq(common["EC_gene"].astype(str))
    & common["MTG_gene"].astype(str).eq(common["SSC_gene"].astype(str))
)
common = common.loc[same_symbol].copy()
common["gene"] = common["MTG_gene"].astype(str)
common = common.loc[~common["gene"].str.startswith("MT-")].copy()
common = common.drop_duplicates("gene")

print("Common unique gene symbols for Hallmark GSEA:", len(common))
common.to_csv(GSE / "SEAAD_vs_GSE160936_common_Ensembl_symbol_GSEA_universe.csv")


In [ ]:
def run_hallmark(label, table):
    out = GSE / f"{label}_commonUniverse_Hallmark_GSEA.csv"
    if not REBUILD and out.exists():
        return pd.read_csv(out)

    rank = table[["gene", f"{label}_stat"]].rename(columns={f"{label}_stat": "stat"})
    rank = rank.dropna().sort_values("stat", ascending=False)

    result = gp.prerank(
        rnk=rank[["gene","stat"]],
        gene_sets="MSigDB_Hallmark_2020",
        min_size=15,
        max_size=500,
        permutation_num=1000,
        seed=42,
        outdir=str(GSE / f"GSEA_{label}_commonUniverse"),
        verbose=False,
    ).res2d.copy()

    result.to_csv(out, index=False)
    return result

gsea = {label: run_hallmark(label, common) for label in ("MTG","DLPFC","EC","SSC")}


In [ ]:
hallmark = None

for label, result in gsea.items():
    x = result[["Term","NES","NOM p-val","FDR q-val"]].copy()
    x = x.rename(columns={
        "NES": f"{label}_NES",
        "NOM p-val": f"{label}_p",
        "FDR q-val": f"{label}_FDR",
    })
    hallmark = x if hallmark is None else hallmark.merge(x, on="Term", how="inner")

hallmark["SEAAD_mean_NES"] = hallmark[["MTG_NES","DLPFC_NES"]].mean(axis=1)
hallmark["GSE_mean_NES"] = hallmark[["EC_NES","SSC_NES"]].mean(axis=1)
hallmark.to_csv(GSE / "SEAAD_vs_GSE160936_HARMONIZED_commonUniverse_Hallmark_concordance.csv", index=False)

same4 = (
    np.sign(hallmark["MTG_NES"]).eq(np.sign(hallmark["DLPFC_NES"]))
    & np.sign(hallmark["MTG_NES"]).eq(np.sign(hallmark["EC_NES"]))
    & np.sign(hallmark["MTG_NES"]).eq(np.sign(hallmark["SSC_NES"]))
)
strict = hallmark.loc[
    same4
    & hallmark["MTG_FDR"].lt(0.05)
    & hallmark["DLPFC_FDR"].lt(0.05)
    & hallmark["EC_FDR"].lt(0.05)
    & hallmark["SSC_FDR"].lt(0.05)
]

print("Strict all-four Hallmark consensus:", len(strict))
print(
    hallmark.loc[hallmark["Term"].eq("Oxidative Phosphorylation"),
                 ["Term","MTG_NES","DLPFC_NES","EC_NES","SSC_NES"]].to_string(index=False)
)


## CREB5 influence audit

In [ ]:
def ols_influence(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, p = X.shape
    xtx_inv = np.linalg.inv(X.T @ X)
    beta = xtx_inv @ X.T @ y
    residuals = y - X @ beta
    mse = np.sum(residuals**2) / (n - p)
    H = X @ xtx_inv @ X.T
    leverage = np.diag(H)
    cooks = (residuals**2 / (p * mse)) * leverage / (1 - leverage)**2
    return beta, cooks


In [ ]:
def creb5_influence_for_region(region, pb, creb5_id):
    dat = pb[pb.obs["region"].eq(region)].copy()
    feature_ids = dat.var["feature_id_unversioned"].astype(str).to_numpy()
    X = dat.X.toarray() if sp.issparse(dat.X) else np.asarray(dat.X)

    counts = pd.DataFrame(
        np.rint(X).astype(np.int64),
        index=pd.Index(dat.obs_names.astype(str), dtype=object),
        columns=pd.Index(feature_ids, dtype=object),
    )
    meta = dat.obs[["donor","disease","RIN","n_astrocytes"]].copy()
    meta["RIN"] = pd.to_numeric(meta["RIN"])
    meta["disease"] = pd.Categorical(
        meta["disease"].astype(str),
        categories=["Non-disease control","AD"],
    )
    meta.index = counts.index

    dds = DeseqDataSet(
        counts=counts, metadata=meta,
        design="~ RIN + disease", refit_cooks=True,
        n_cpus=2, quiet=True,
    )
    dds.fit_size_factors()
    size_factor = dds.obs["size_factors"].to_numpy(dtype=float)

    raw_creb = counts[creb5_id].to_numpy(dtype=float)
    norm_creb = raw_creb / size_factor
    y = np.log2(norm_creb + 1)

    disease = meta["disease"].astype(str).eq("AD").astype(float).to_numpy()
    rin = meta["RIN"].to_numpy(dtype=float)
    design = np.column_stack([np.ones(len(meta)), rin, disease])
    _, cooks = ols_influence(design, y)

    loo = []
    for i in range(len(meta)):
        keep = np.arange(len(meta)) != i
        beta_i, _ = ols_influence(design[keep], y[keep])
        loo.append({
            "region": region,
            "left_out_GSM": meta.index[i],
            "left_out_donor": meta.iloc[i]["donor"],
            "left_out_disease": str(meta.iloc[i]["disease"]),
            "disease_beta": float(beta_i[2]),
        })

    tab = meta.reset_index(names="GSM")
    tab["region"] = region
    tab["CREB5_raw_count"] = raw_creb.astype(int)
    tab["size_factor"] = size_factor
    tab["CREB5_normalized"] = norm_creb
    tab["CREB5_log2_norm_plus1"] = y
    tab["CookD_exploratory"] = cooks
    return tab, pd.DataFrame(loo)


In [ ]:
CREB5_ID = "ENSG00000146592"
norm_path = GSE / "GSE160936_CREB5_normalized_expression_influence_audit.csv"
loo_path = GSE / "GSE160936_CREB5_leave_one_donor_out_influence_audit.csv"

if REBUILD or not (norm_path.exists() and loo_path.exists()):
    pb = ad.read_h5ad(de_universe_path)
    sample_tables, loo_tables = [], []

    for region in ("EC", "SSC"):
        tab, loo = creb5_influence_for_region(region, pb, CREB5_ID)
        sample_tables.append(tab)
        loo_tables.append(loo)

    norm_audit = pd.concat(sample_tables, ignore_index=True)
    loo_audit = pd.concat(loo_tables, ignore_index=True)
    norm_audit.to_csv(norm_path, index=False)
    loo_audit.to_csv(loo_path, index=False)
else:
    norm_audit = pd.read_csv(norm_path)
    loo_audit = pd.read_csv(loo_path)

for region in ("EC", "SSC"):
    x = loo_audit.loc[loo_audit["region"].eq(region), "disease_beta"].to_numpy()
    print(region, "| LOO range:", f"{x.min():.4f} to {x.max():.4f}", "| all positive:", bool((x > 0).all()))


## Primary CREB5 estimates

In [ ]:
rows = []
for cohort, region, path in [
    ("SEA-AD","MTG", ROOT / "SEAAD_MTG_High_vs_NotAD_DESeq2_primary_RINadjusted.csv"),
    ("SEA-AD","DLPFC", ROOT / "SEAAD_DLPFC_High_vs_NotAD_DESeq2_primary_RINadjusted.csv"),
    ("GSE160936","EC", GSE / "GSE160936_EC_AD_vs_Control_DESeq2_primary_RINadjusted.csv"),
    ("GSE160936","SSC", GSE / "GSE160936_SSC_AD_vs_Control_DESeq2_primary_RINadjusted.csv"),
]:
    res = pd.read_csv(path, index_col=0)
    row = res.loc[res["gene"].eq("CREB5")].iloc[0]
    rows.append({
        "cohort": cohort,
        "region": region,
        "log2FC": row["log2FoldChange"],
        "SE": row["lfcSE"],
        "CI_low": row["log2FoldChange"] - 1.96 * row["lfcSE"],
        "CI_high": row["log2FoldChange"] + 1.96 * row["lfcSE"],
        "P": row["pvalue"],
        "FDR": row["padj"],
    })

creb5_primary = pd.DataFrame(rows)
creb5_primary.to_csv(GSE / "CREB5_four_region_primary_summary.csv", index=False)
print(creb5_primary.to_string(index=False))


### Notes

- SEA-AD MTG and DLPFC are regional analyses from the same atlas and are not treated as independent studies.
- GSE160936 EC and SSC are paired tissues from the same 12 donors and are analyzed separately because RIN is region-specific.
- The OLS/Cook's-distance calculations are influence diagnostics only; the reported differential-expression estimates come from the donor-level negative-binomial models above.
